# Daikin Park Game & Weather Dataset (2017-2025)

This notebook builds a comprehensive dataset where each row represents a regular season game played at Daikin Park (formerly Minute Maid Park). It combines:
1. **Game statistics** from `master_data.csv` (pre-aggregated Statcast data)
2. **Weather data** from Open-Meteo hourly observations
3. **Wind projections** onto outfield vectors (CF, LCF, RCF)

All batting/pitching statistics are **both teams combined** to capture the full park-environment effect.

**Note**: Daikin Park has a retractable roof, so outdoor weather conditions may not fully reflect in-stadium conditions when the roof is closed. The weather data is still included as it captures ambient atmospheric conditions (temperature, pressure, humidity) that affect air density and ball flight regardless of roof status.

## Section 0: Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import requests
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# === STADIUM CONFIGURATION ===
STADIUM_NAME = 'Daikin Park'
HOME_TEAM = 'HOU'
SEASONS = range(2017, 2026)  # 2017 through 2025
TIMEZONE = 'America/Chicago'

# Coordinates
STADIUM_LAT = 29.756548
STADIUM_LON = -95.355532

# Outfield directions (degrees from north)
CF_DIR = 337.5   # Center field: NNW
LCF_DIR = 317.5  # Left-center field: NW
RCF_DIR = 357.5  # Right-center field: N

# Output file
OUTPUT_FILE = 'astros_data_2017.csv'

print(f"Configuration: {STADIUM_NAME}")
print(f"Home team: {HOME_TEAM}")
print(f"Seasons: {list(SEASONS)}")
print(f"Timezone: {TIMEZONE}")

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


Configuration: Daikin Park
Home team: HOU
Seasons: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Timezone: America/Chicago


## Section 1: Load & Filter Master Data

Read game-level statistics from `master_data.csv` and filter to Daikin Park home games.

In [2]:
# Load master dataset
master = pd.read_csv(os.path.join('..', 'Final Datasets', 'master_data.csv'))
print(f"Master dataset: {len(master)} total games")

# Filter to this stadium's home games and season range
games = master[
    (master['home_team'] == HOME_TEAM) &
    (master['season'].isin(SEASONS))
].copy()

# Convert game_start_utc to local timezone
games['game_start'] = (
    pd.to_datetime(games['game_start_utc'], utc=True)
    .dt.tz_convert(TIMEZONE)
    .dt.tz_localize(None)  # Remove timezone info for clean processing
)
games['start_hour'] = games['game_start'].dt.hour
games['game_date'] = pd.to_datetime(games['game_date'])

# Drop master-only columns not needed in final output
games = games.drop(columns=['home_team', 'game_start_utc'])

games = games.sort_values('game_date').reset_index(drop=True)

print(f"\n{STADIUM_NAME} games: {len(games)}")
print(f"Seasons: {sorted(games['season'].unique())}")
print(f"\nGames per season:")
print(games.groupby('season')['game_pk'].count())
print(f"\nSample:")
print(games.head(3))

Master dataset: 25155 total games

Daikin Park games: 677
Seasons: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Games per season:
season
2017    81
2018    81
2019    81
2020    30
2021    81
2022    81
2023    81
2024    81
2025    80
Name: game_pk, dtype: int64

Sample:
   game_pk  game_date  season away_team  home_runs_scored  away_runs_scored  total_runs  home_runs_hit  strikeouts  walks  hits  total_pitches  avg_exit_velocity  n_barrels  n_bbe  barrel_rate  \
0   490111 2017-04-03    2017       SEA                 3                 0           3              2          14      5     9            226               83.7          2     43       0.0465   
1   490119 2017-04-04    2017       SEA                 2                 1           3              2          14      5    12            240               81.9          0     48       0.0000   
2   490133 2017-04-05    2017       SEA                 5                 3           8              2          23     11    19 

## Section 2: Pull Weather Data (Open-Meteo)

Use Open-Meteo to pull hourly weather data for each season, then average over the 3 hours following each game's start time.

In [3]:
# Using Open-Meteo Historical Weather API (free, no key required)
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_PARAMS = "temperature_2m,relative_humidity_2m,surface_pressure,precipitation,wind_speed_10m,wind_direction_10m"

# Test fetch to confirm API is reachable
test_resp = requests.get(OPEN_METEO_URL, params={
    'latitude': STADIUM_LAT,
    'longitude': STADIUM_LON,
    'start_date': '2023-07-01',
    'end_date': '2023-07-02',
    'hourly': HOURLY_PARAMS,
    'timezone': TIMEZONE,
})

if test_resp.status_code == 200:
    test_data = test_resp.json()
    n_hours = len(test_data['hourly']['time'])
    print(f"Open-Meteo API test (Jul 1-2 2023): {n_hours} hourly records - OK")
    print(f"Sample time: {test_data['hourly']['time'][12]}")
    print(f"Sample temp: {test_data['hourly']['temperature_2m'][12]}\u00b0C")
    print(f"Sample wind: {test_data['hourly']['wind_speed_10m'][12]} km/h from {test_data['hourly']['wind_direction_10m'][12]}\u00b0")
else:
    print(f"ERROR: Open-Meteo API returned {test_resp.status_code}")
    print(test_resp.text)

Open-Meteo API test (Jul 1-2 2023): 48 hourly records - OK
Sample time: 2023-07-01T12:00
Sample temp: 35.0°C
Sample wind: 9.0 km/h from 185°


In [4]:
def fetch_season_weather(year):
    """Fetch hourly weather for a full season from Open-Meteo."""
    resp = requests.get(OPEN_METEO_URL, params={
        'latitude': STADIUM_LAT,
        'longitude': STADIUM_LON,
        'start_date': f'{year}-03-01',
        'end_date': f'{year}-11-30',
        'hourly': HOURLY_PARAMS,
        'timezone': TIMEZONE,
    })
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    
    df = pd.DataFrame({
        'temp': hourly['temperature_2m'],
        'rhum': hourly['relative_humidity_2m'],
        'pres': hourly['surface_pressure'],
        'prcp': hourly['precipitation'],
        'wspd': hourly['wind_speed_10m'],
        'wdir': hourly['wind_direction_10m'],
    }, index=pd.to_datetime(hourly['time']))
    
    return df


def get_game_weather(game_start_dt, hourly_df):
    """
    Average weather over the 3 hours following game start.
    game_start_dt: datetime (local time, rounded to hour)
    hourly_df: DataFrame with hourly weather, index is naive local time
    """
    start = game_start_dt
    end = start + timedelta(hours=2)  # 3 hourly obs: start, +1h, +2h
    
    window = hourly_df.loc[start:end]
    
    if len(window) == 0:
        return pd.Series({
            'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
            'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
        })
    
    result = {
        'temp_c': window['temp'].mean(),
        'rhum': window['rhum'].mean(),
        'pres': window['pres'].mean(),
        'prcp': window['prcp'].sum(),   # Precipitation SUMMED (cumulative quantity)
        'wspd': window['wspd'].mean(),
    }
    
    # Wind direction: circular mean to handle 0/360 boundary
    wdir_vals = window['wdir'].dropna()
    if len(wdir_vals) > 0:
        wdir_rad = np.radians(wdir_vals)
        mean_sin = np.sin(wdir_rad).mean()
        mean_cos = np.cos(wdir_rad).mean()
        result['wdir'] = np.degrees(np.arctan2(mean_sin, mean_cos)) % 360
    else:
        result['wdir'] = np.nan
    
    return pd.Series(result)


# Pull weather season by season
weather_records = []

for year in SEASONS:
    print(f"Pulling weather for {year}...")
    
    try:
        hourly_df = fetch_season_weather(year)
    except Exception as e:
        print(f"  WARNING: Failed for {year}: {e}")
        season_games = games[games['season'] == year]
        for idx, game in season_games.iterrows():
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
        continue
    
    print(f"  {year}: {len(hourly_df)} hourly records")
    
    season_games = games[games['season'] == year]
    for idx, game in season_games.iterrows():
        if pd.isna(game['game_start']):
            weather_records.append({
                'game_pk': game['game_pk'],
                'temp_c': np.nan, 'rhum': np.nan, 'pres': np.nan,
                'prcp': np.nan, 'wspd': np.nan, 'wdir': np.nan
            })
            continue
        
        game_hour = game['game_start'].replace(minute=0, second=0, microsecond=0)
        wx = get_game_weather(game_hour, hourly_df)
        wx['game_pk'] = game['game_pk']
        weather_records.append(wx.to_dict())

weather_df = pd.DataFrame(weather_records)
weather_df['game_pk'] = weather_df['game_pk'].astype('Int64')
print(f"\nWeather records: {len(weather_df)}")
print(f"Missing temp data: {weather_df['temp_c'].isna().sum()}")
print(weather_df.head(3))

Pulling weather for 2017...
  2017: 6600 hourly records
Pulling weather for 2018...
  2018: 6600 hourly records
Pulling weather for 2019...
  2019: 6600 hourly records
Pulling weather for 2020...
  2020: 6600 hourly records
Pulling weather for 2021...
  2021: 6600 hourly records
Pulling weather for 2022...
  2022: 6600 hourly records
Pulling weather for 2023...
  2023: 6600 hourly records
Pulling weather for 2024...
  2024: 6600 hourly records
Pulling weather for 2025...
  2025: 6600 hourly records

Weather records: 677
Missing temp data: 0
      temp_c       rhum         pres  prcp       wspd        wdir  game_pk
0  20.933333  73.666667  1003.600000   0.0  10.733333  199.993885   490111
1  22.533333  88.666667  1003.333333   0.0  13.700000  159.610716   490119
2  16.600000  47.000000  1016.000000   0.0  12.266667  333.342683   490133


## Section 3: Wind Direction Bucketing & Outfield Projections

**Wind direction bucketing**: 8 compass directions (N, NE, E, SE, S, SW, W, NW).

**Wind projections**: Project wind onto vectors from home plate to center field (CF), left-center field (LCF), and right-center field (RCF). Positive = blowing out, negative = blowing in.

Daikin Park outfield directions (degrees from north):
- Center field: ~337.5\u00b0 (NNW)
- Left-center field: ~317.5\u00b0 (NW)
- Right-center field: ~357.5\u00b0 (N)

**Important**: Weather APIs report wind direction as the direction wind blows **FROM**. We must convert to the direction it blows **TO** before projecting.

In [5]:
# Merge weather into game data
games_full = games.merge(weather_df, on='game_pk', how='left')

# --- Wind direction bucketing ---
def bucket_wind_dir(deg):
    """Bucket wind direction (degrees) into 8 compass directions."""
    if pd.isna(deg):
        return np.nan
    buckets = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int(((deg + 22.5) % 360) / 45)
    return buckets[idx]

games_full['wind_dir_bucket'] = games_full['wdir'].apply(bucket_wind_dir)

# --- Wind projections onto outfield vectors ---
def compute_wind_projection(wdir, wspd, outfield_dir):
    """
    Project wind onto an outfield direction vector.
    
    wdir: direction wind blows FROM (meteorological convention, degrees)
    wspd: wind speed (km/h)
    outfield_dir: compass bearing from home plate to outfield (degrees from north)
    
    Returns: positive = blowing OUT toward outfield, negative = blowing IN
    """
    if pd.isna(wdir) or pd.isna(wspd):
        return np.nan
    # Wind blows FROM wdir, so it travels TOWARD (wdir + 180)
    wind_toward = (wdir + 180) % 360
    # Project onto outfield direction
    angle_diff = wind_toward - outfield_dir
    return wspd * np.cos(np.radians(angle_diff))

games_full['wind_cf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], CF_DIR), axis=1
)
games_full['wind_lcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], LCF_DIR), axis=1
)
games_full['wind_rcf'] = games_full.apply(
    lambda r: compute_wind_projection(r['wdir'], r['wspd'], RCF_DIR), axis=1
)

print("Wind projection summary (positive = blowing out, negative = blowing in):")
print(games_full[['wind_cf', 'wind_lcf', 'wind_rcf']].describe())

Wind projection summary (positive = blowing out, negative = blowing in):
          wind_cf    wind_lcf    wind_rcf
count  677.000000  677.000000  677.000000
mean     7.889348    8.065798    6.761327
std      9.388610    8.654187    9.450189
min    -31.510561  -32.070556  -32.599242
25%      3.754490    4.172716    2.578935
50%      8.821005    8.887951    7.967334
75%     13.892464   13.812799   12.900923
max     31.395477   33.366448   31.436694


## Section 4: Final Assembly

Convert units, order columns, and round to sensible precision.

In [6]:
# Unit conversions
games_full['temp_f'] = games_full['temp_c'] * 9/5 + 32
games_full['wspd_mph'] = games_full['wspd'] * 0.621371

# Final column order
final_columns = [
    # Game identification
    'game_pk', 'game_date', 'season', 'away_team', 'game_start', 'start_hour',
    # Scoring
    'home_runs_scored', 'away_runs_scored', 'total_runs',
    # Batting stats (both teams combined)
    'home_runs_hit', 'strikeouts', 'walks', 'hits',
    'total_pitches', 'avg_exit_velocity',
    'n_barrels', 'n_bbe', 'barrel_rate', 'hr_h_ratio',
    # Weather
    'temp_f', 'temp_c', 'rhum', 'pres', 'prcp',
    'wspd', 'wspd_mph', 'wdir', 'wind_dir_bucket',
    # Wind projections
    'wind_cf', 'wind_lcf', 'wind_rcf',
]

astros_data = games_full[final_columns].copy()
astros_data = astros_data.sort_values('game_date').reset_index(drop=True)

# Round floating point columns
round_map = {
    'avg_exit_velocity': 1, 'barrel_rate': 4, 'hr_h_ratio': 4,
    'temp_f': 1, 'temp_c': 1, 'rhum': 1, 'pres': 1, 'prcp': 2,
    'wspd': 1, 'wspd_mph': 1, 'wdir': 1,
    'wind_cf': 2, 'wind_lcf': 2, 'wind_rcf': 2,
}
for col, decimals in round_map.items():
    astros_data[col] = astros_data[col].round(decimals)

print(f"Final dataset: {astros_data.shape[0]} rows x {astros_data.shape[1]} columns")

Final dataset: 677 rows x 31 columns


## Section 5: Validation

Verify row counts per season, check for nulls, and sanity-check summary statistics.

In [7]:
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# 1. Row counts per season
print("\n--- Games per Season ---")
season_counts = astros_data.groupby('season').size()
for year, count in season_counts.items():
    if year == 2020:
        expected = (25, 35)  # COVID shortened season
    else:
        expected = (75, 100)  # Normal: ~81 home games
    status = "OK" if expected[0] <= count <= expected[1] else "WARNING"
    print(f"  {year}: {count} games [{status}] (expected {expected[0]}-{expected[1]})")
print(f"  TOTAL: {len(astros_data)} games")

# 2. Null check
print("\n--- Null Counts ---")
key_cols = ['total_runs', 'home_runs_hit', 'strikeouts', 'walks',
            'total_pitches', 'avg_exit_velocity', 'barrel_rate',
            'temp_f', 'wspd', 'wdir', 'wind_cf', 'game_start']
for col in key_cols:
    n_null = astros_data[col].isna().sum()
    pct = 100 * n_null / len(astros_data)
    status = "OK" if pct < 5 else "WARNING"
    print(f"  {col}: {n_null} nulls ({pct:.1f}%) [{status}]")

# 3. Summary statistics sanity checks
print("\n--- Sanity Checks ---")
checks = [
    ('Avg total runs/game', astros_data['total_runs'].mean(), '~8-10'),
    ('Avg HR/game', astros_data['home_runs_hit'].mean(), '~2-3'),
    ('Avg K/game', astros_data['strikeouts'].mean(), '~16-18'),
    ('Avg BB/game', astros_data['walks'].mean(), '~6-7'),
    ('Avg exit velocity', astros_data['avg_exit_velocity'].mean(), '~87-89 mph'),
    ('Avg barrel rate', astros_data['barrel_rate'].mean(), '~0.06-0.08'),
    ('Avg game temp', astros_data['temp_f'].mean(), '~75-90 F'),
    ('Min game temp', astros_data['temp_f'].min(), '>40 F'),
    ('Max game temp', astros_data['temp_f'].max(), '<105 F'),
    ('Avg wind speed (km/h)', astros_data['wspd'].mean(), '~8-18 km/h'),
]
for label, val, expected in checks:
    print(f"  {label}: {val:.2f} (expected {expected})")

# 4. Full summary statistics
print("\n--- Summary Statistics ---")
print(astros_data.describe().T[['mean', 'std', 'min', 'max']].to_string())

VALIDATION REPORT

--- Games per Season ---
  2017: 81 games [OK] (expected 75-100)
  2018: 81 games [OK] (expected 75-100)
  2019: 81 games [OK] (expected 75-100)
  2020: 30 games [OK] (expected 25-35)
  2021: 81 games [OK] (expected 75-100)
  2022: 81 games [OK] (expected 75-100)
  2023: 81 games [OK] (expected 75-100)
  2024: 81 games [OK] (expected 75-100)
  2025: 80 games [OK] (expected 75-100)
  TOTAL: 677 games

--- Null Counts ---
  total_runs: 0 nulls (0.0%) [OK]
  home_runs_hit: 0 nulls (0.0%) [OK]
  strikeouts: 0 nulls (0.0%) [OK]
  walks: 0 nulls (0.0%) [OK]
  total_pitches: 0 nulls (0.0%) [OK]
  avg_exit_velocity: 0 nulls (0.0%) [OK]
  barrel_rate: 0 nulls (0.0%) [OK]
  temp_f: 0 nulls (0.0%) [OK]
  wspd: 0 nulls (0.0%) [OK]
  wdir: 0 nulls (0.0%) [OK]
  wind_cf: 0 nulls (0.0%) [OK]
  game_start: 0 nulls (0.0%) [OK]

--- Sanity Checks ---
  Avg total runs/game: 8.77 (expected ~8-10)
  Avg HR/game: 2.55 (expected ~2-3)
  Avg K/game: 17.16 (expected ~16-18)
  Avg BB/game: 6.

In [8]:
# Print dataset header for inspection
print("\n--- First 10 Rows ---")
astros_data.head(10)


--- First 10 Rows ---


,game_pk,game_date,season,away_team,game_start,start_hour,home_runs_scored,away_runs_scored,total_runs,home_runs_hit,strikeouts,walks,hits,total_pitches,avg_exit_velocity,n_barrels,n_bbe,barrel_rate,hr_h_ratio,temp_f,temp_c,rhum,pres,prcp,wspd,wspd_mph,wdir,wind_dir_bucket,wind_cf,wind_lcf,wind_rcf
0,490111,2017-04-03,2017,SEA,2017-04-03 19:10:00,19,3,0,3,2,14,5,9,226,83.7,2,43,0.0465,0.2222,69.7,20.9,73.7,1003.6,0.0,10.7,6.7,200.0,S,7.91,4.96,9.92
1,490119,2017-04-04,2017,SEA,2017-04-04 19:10:00,19,2,1,3,2,14,5,12,240,81.9,0,48,0.0000,0.1667,72.6,22.5,88.7,1003.3,0.0,13.7,8.5,159.6,S,13.69,12.69,13.04
2,490133,2017-04-05,2017,SEA,2017-04-05 19:10:00,19,5,3,8,2,23,11,19,422,80.1,3,74,0.0405,0.1053,61.9,16.6,47.0,1016.0,0.0,12.3,7.6,333.3,NW,-12.23,-11.80,-11.19
3,490146,2017-04-06,2017,SEA,2017-04-06 19:10:00,19,2,4,6,3,18,7,13,286,82.8,4,47,0.0851,0.2308,64.7,18.2,53.7,1018.7,0.0,8.1,5.0,68.7,E,0.17,2.93,-2.61
4,490153,2017-04-07,2017,KC,2017-04-07 19:10:00,19,1,5,6,1,13,9,16,306,80.4,2,52,0.0385,0.0625,64.9,18.3,61.7,1017.7,0.0,11.8,7.3,180.7,S,10.82,8.58,11.75
5,490168,2017-04-08,2017,KC,2017-04-08 18:10:00,18,3,7,10,4,11,4,16,244,83.7,3,55,0.0545,0.2500,68.8,20.4,65.0,1012.7,0.0,15.8,9.8,160.3,S,15.78,14.56,15.10
6,490183,2017-04-09,2017,KC,2017-04-09 13:10:00,13,5,4,9,6,24,8,21,366,82.5,5,62,0.0806,0.2857,77.4,25.2,59.7,1011.6,0.0,27.5,17.1,159.3,S,27.45,25.50,26.10
7,490288,2017-04-17,2017,LAA,2017-04-17 19:10:00,19,3,0,3,0,15,4,14,253,82.6,1,47,0.0213,0.0000,68.0,20.0,95.7,1013.2,4.8,6.8,4.2,40.3,NE,-3.12,-0.85,-5.01
8,490302,2017-04-18,2017,LAA,2017-04-18 19:10:00,19,2,5,7,4,18,4,20,300,82.0,2,52,0.0385,0.2000,69.9,21.0,91.3,1013.8,0.4,11.7,7.2,112.8,SE,8.29,10.60,4.98
9,490317,2017-04-19,2017,LAA,2017-04-19 19:10:00,19,5,1,6,1,16,3,17,245,80.9,1,46,0.0217,0.0588,72.4,22.5,87.7,1014.7,0.0,11.3,7.0,138.6,SE,10.69,11.30,8.79


## Section 6: Save to CSV

In [9]:
# Save final dataset
output_dir = os.path.join('..', 'Final Datasets')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, OUTPUT_FILE)
astros_data.to_csv(output_path, index=False)

print(f"Saved to: {os.path.abspath(output_path)}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Rows: {len(astros_data)}, Columns: {len(astros_data.columns)}")

# Verify roundtrip
verify = pd.read_csv(output_path)
assert verify.shape == astros_data.shape, f"Shape mismatch: {verify.shape} vs {astros_data.shape}"
print("\nSave & reload verification: PASSED")

Saved to: /Users/avabrown/Desktop/DATASCI 192A/Stadium Datasets/Final Datasets/astros_data_2017.csv
File size: 101.6 KB
Rows: 677, Columns: 31

Save & reload verification: PASSED
